# Install packages



In [1]:
!pip install pygamma-agreement "pygamma-agreement[CBC]" pyannote.core pandas numpy

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 98.1 MB/s eta 0:00:00
  Created wheel for TextGrid: filename=TextGrid-1.6.1-py3-none-any.whl size=10146 sha256=72a29324c9767ef5eda88570245b007a1a915bea2789f59b8fb9dc666e669ca4
  Stored in directory: /root/.cache/pip/wheels/ce/86/7b/5766bd19fa4b4554667dd186e614b5a438ab14eec9c5a3642a
Successfully built TextGrid
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not curr

# mount Google Drive



In [2]:
# Run this only in Google Colab.
# In VS Code on your laptop, skip it.

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Not running in Colab, so Google Drive was not mounted.")

Mounted at /content/drive


# Imports

In [3]:
import json
import re
from pathlib import Path
from itertools import combinations
from typing import Optional

import numpy as np
import pandas as pd

from pyannote.core import Segment
from pygamma_agreement import Continuum, CombinedCategoricalDissimilarity

# Main folder configuration

Edit `PROJECT_ROOT` to the folder where your data is stored.

Expected structure:

```text
PROJECT_ROOT/
  raw_texts/
    TED_014.txt
    TED_013.txt
  annotations/
    TED_014/
      filip.json
      nabhani.json
  llm_outputs/
    TED_014_mistral-7b_prompt-v2_argumentation.json
    TED_014_qwen-7b_prompt-v2_argumentation.json
    TED_014_gemma-7b_prompt-v2_argumentation.json
  gamma_results/
```

In [ ]:
RAW_TEXT_ROOT = PROJECT_ROOT / "raw_texts"
ANNOTATION_ROOT = PROJECT_ROOT / "annotations"
LLM_OUTPUT_ROOT = PROJECT_ROOT / "llm_outputs"
OUTPUT_ROOT = PROJECT_ROOT / "gamma_results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DOCUMENT_IDS = ["TED_001", "TED_002", "TED_004", "TED_005", "TED_006", "TED_007", "TED_008", "TED_010", "TED_011", "TED_012", "TED_015", "TED_016", "TED_017", "TED_018", "TED_019", "TED_020"]

HUMAN_ANNOTATORS = {
    "Luiza": "filip.json",
    "Sara": "nabhani.json",
}

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
TASKS = {
    #"argumentation": ["C-EXP", "MC-EXP", "P-EXP"],
    # Uncomment/edit if you also want narration Gamma:
    "narration": ["Confirmatio", "Narratio", "Exordium", "Other", "Peroratio", "Propositio"],
}

MODELS = {
    "Mistral": {
        "model_name": "mistral-small-3.2-24b-instruct",
        "prompt_version": "final",
    },
    "Qwen": {
        "model_name": "qwen3.6-27b",
        "prompt_version": "final",
    },
    "Gemma": {
        "model_name": "gemma-4-31b-it",
        "prompt_version": "final",
    },
}

MODEL_PAIRS = [
    ("Mistral", "Qwen"),
    ("Mistral", "Gemma"),
    ("Qwen", "Gemma"),
]

# Filename helper

Change this function only if your LLM output filenames use a different naming pattern.

In [24]:
def llm_filename(document_id, model_name, prompt_version, task_type):
    return f"{document_id}_{model_name}_prompt-{prompt_version}_{task_type}.json"

# Check that folders/files exist



In [25]:
def check_basic_paths():
    print("PROJECT_ROOT:", PROJECT_ROOT)
    print("Exists:", PROJECT_ROOT.exists())
    print()

    for folder in [RAW_TEXT_ROOT, ANNOTATION_ROOT, LLM_OUTPUT_ROOT, OUTPUT_ROOT]:
        print(folder, "exists =", folder.exists())

    print("Checking document files:")
    for document_id in DOCUMENT_IDS:
        print(document_id)
        print("  text:", (RAW_TEXT_ROOT / f"{document_id}.txt").exists(), RAW_TEXT_ROOT / f"{document_id}.txt")
        for human_name, human_file in HUMAN_ANNOTATORS.items():
            print(f"  human {human_name}:", (ANNOTATION_ROOT / document_id / human_file).exists(), ANNOTATION_ROOT / document_id / human_file)
        for task_type in TASKS:
            for model_label, info in MODELS.items():
                path = LLM_OUTPUT_ROOT / llm_filename(document_id, info["model_name"], info["prompt_version"], task_type)
                print(f"  model {model_label} {task_type}:", path.exists(), path)
        print()

check_basic_paths()

PROJECT_ROOT: /content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL
Exists: True

/content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/raw_texts exists = True
/content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/annotations exists = True
/content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/llm_outputs exists = True
/content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/gamma_results exists = True
Checking document files:
TED_001
  text: True /content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/raw_texts/TED_001.txt
  human Luiza: True /content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/annotations/TED_001/filip.json
  human Sara: True /content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/annotations/TED_001/nabhani.json
  model Mistral narration: True /content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/llm_outputs/TED_001_mistral-small-3.2-24b-instruct_pr

## Parse human INCEpTION annotations

In [26]:
def parse_inception_json(filepath, annotator_name):
    """Parse human INCEpTION JSON into Gamma-ready annotations."""
    filepath = Path(filepath)

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    feature_structures = data["%FEATURE_STRUCTURES"]

    feature_name_to_ui_name = {}
    for fs in feature_structures:
        if fs.get("%TYPE") == "de.tudarmstadt.ukp.clarin.webanno.api.type.FeatureDefinition":
            internal_name = fs.get("name")
            readable_name = fs.get("uiName")
            if internal_name and readable_name:
                feature_name_to_ui_name[internal_name] = readable_name

    ignore_features = {"IClaimtext", "MCIMPtext", "PIMPtext"}

    annotation_layers = {
        #"webanno.custom.Argumentation",
        "webanno.custom.Narration",
    }

    annotations = []

    for fs in feature_structures:
        if fs.get("%TYPE") not in annotation_layers:
            continue

        start = fs.get("begin")
        end = fs.get("end")

        if start is None or end is None:
            continue

        for feature_name, value in fs.items():
            if feature_name.startswith("%") or feature_name.startswith("@"):
                continue
            if feature_name in {"begin", "end"}:
                continue
            if feature_name in ignore_features:
                continue

            if value is True:
                annotations.append({
                    "annotator": annotator_name,
                    "start": int(start),
                    "end": int(end),
                    "label": feature_name_to_ui_name.get(feature_name, feature_name),
                })

    return annotations

## Parse LLM outputs and convert text spans to offsets



In [27]:
def read_llm_output(filepath):
    """Read LLM output JSON, even if wrapped in ```json fences."""
    filepath = Path(filepath)

    with open(filepath, "r", encoding="utf-8") as f:
        output = f.read().strip()

    output = re.sub(r"```json", "", output, flags=re.IGNORECASE)
    output = re.sub(r"```", "", output)

    start = output.find("{")
    end = output.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in {filepath}")

    data = json.loads(output[start:end + 1])

    if isinstance(data, list):
        return data

    if "annotations" in data:
        return data["annotations"]

    if "argumentation" in data or "narration" in data:
        return data.get("argumentation", []) + data.get("narration", [])

    raise ValueError(f"Unknown LLM output format in {filepath}")


def normalize_text_for_matching(text):
    return re.sub(r"\s+", " ", text).strip()


def find_span_offsets(ted_text, span_text, search_from=0):
    """
    Find span offsets in the TED text.
    First tries exact matching. Then tries whitespace-normalized matching.
    """
    if not span_text:
        return None

    start = ted_text.find(span_text, search_from)
    if start == -1:
        start = ted_text.find(span_text)

    if start != -1:
        return start, start + len(span_text)

    # Fallback: whitespace-normalized matching.
    normalized_span = normalize_text_for_matching(span_text)
    if not normalized_span:
        return None

    pattern = re.escape(normalized_span)
    pattern = pattern.replace(r"\ ", r"\s+")

    match = re.search(pattern, ted_text[search_from:], flags=re.IGNORECASE)
    if match:
        start = search_from + match.start()
        end = search_from + match.end()
        return start, end

    match = re.search(pattern, ted_text, flags=re.IGNORECASE)
    if match:
        return match.start(), match.end()

    return None


def llm_text_to_offsets(llm_annotations, ted_text, annotator_name):
    """Convert LLM text spans to character offsets."""
    converted = []
    search_from = 0

    for ann in llm_annotations:
        span_text = ann.get("text") or ann.get("span") or ann.get("quote")
        label = ann.get("label") or ann.get("category") or ann.get("type")

        # If your model already outputs offsets, use them.
        start_offset = ann.get("start") or ann.get("begin")
        end_offset = ann.get("end")

        if label is None:
            continue

        if start_offset is not None and end_offset is not None:
            start = int(start_offset)
            end = int(end_offset)
        else:
            found = find_span_offsets(ted_text, span_text, search_from=search_from)
            if found is None:
                print(f"WARNING: Could not find LLM span for {annotator_name}: {str(span_text)[:100]!r}")
                continue
            start, end = found

        search_from = end

        converted.append({
            "annotator": annotator_name,
            "start": int(start),
            "end": int(end),
            "label": str(label),
        })

    return converted


def filter_annotations(annotations, selected_labels):
    return [ann for ann in annotations if ann["label"] in selected_labels]

## Load annotations for one document/task

In [28]:
def load_text(document_id):
    text_path = RAW_TEXT_ROOT / f"{document_id}.txt"
    with open(text_path, "r", encoding="utf-8") as f:
        return f.read()


def load_human_annotations(document_id, human_name, labels):
    human_file = HUMAN_ANNOTATORS[human_name]
    human_path = ANNOTATION_ROOT / document_id / human_file

    annotations = parse_inception_json(human_path, human_name)
    return filter_annotations(annotations, labels)


def load_model_annotations(document_id, model_label, task_type, labels):
    info = MODELS[model_label]
    model_name = info["model_name"]
    prompt_version = info["prompt_version"]

    ted_text = load_text(document_id)

    llm_path = LLM_OUTPUT_ROOT / llm_filename(
        document_id=document_id,
        model_name=model_name,
        prompt_version=prompt_version,
        task_type=task_type,
    )

    raw_annotations = read_llm_output(llm_path)
    annotations = llm_text_to_offsets(raw_annotations, ted_text, annotator_name=model_label)
    return filter_annotations(annotations, labels)

## Cell 11 — Gamma calculation functions

This runs the same four Gamma variants:

- standard Gamma, alpha=1, beta=1
- soft Gamma, alpha=1, beta=1
- standard Gamma, alpha=1, beta=2
- soft Gamma, alpha=1, beta=2

In [29]:
def compute_gamma_test(
    annotations,
    layer_name,
    selected_labels=None,
    alpha=1,
    beta=1,
    soft=False,
    seed=42,
    expected_annotators=None,
    add_none_if_missing=True,
):
    """Compute Gamma for two or more annotators."""
    selected_annotations = [
        ann for ann in annotations
        if selected_labels is None or ann["label"] in selected_labels
    ]

    if expected_annotators is None:
        expected_annotators = {ann["annotator"] for ann in annotations}
    else:
        expected_annotators = set(expected_annotators)

    if add_none_if_missing:
        present = {ann["annotator"] for ann in selected_annotations}
        missing = expected_annotators - present

        for annotator in missing:
            selected_annotations.append({
                "annotator": annotator,
                "start": 0,
                "end": 1,
                "label": "none",
            })

    continuum = Continuum()

    for ann in selected_annotations:
        if ann["end"] <= ann["start"]:
            continue

        continuum.add(
            ann["annotator"],
            Segment(ann["start"], ann["end"]),
            ann["label"],
        )

    dissimilarity = CombinedCategoricalDissimilarity(alpha=alpha, beta=beta)
    np.random.seed(seed)

    gamma_result = continuum.compute_gamma(dissimilarity, soft=soft)

    return {
        "layer": layer_name,
        "gamma_type": "soft" if soft else "standard",
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma_result.gamma,
    }


def gamma_suite(annotations, labels, layer_name, expected_annotators):
    """Run the four Gamma variants."""
    settings = [
        {"alpha": 1, "beta": 1, "soft": False},
        {"alpha": 1, "beta": 1, "soft": True},
        {"alpha": 1, "beta": 2, "soft": False},
        {"alpha": 1, "beta": 2, "soft": True},
    ]

    results = []
    for s in settings:
        try:
            result = compute_gamma_test(
                annotations=annotations,
                layer_name=layer_name,
                selected_labels=labels,
                alpha=s["alpha"],
                beta=s["beta"],
                soft=s["soft"],
                expected_annotators=expected_annotators,
            )
        except Exception as e:
            result = {
                "layer": layer_name,
                "gamma_type": "soft" if s["soft"] else "standard",
                "alpha": s["alpha"],
                "beta": s["beta"],
                "gamma": np.nan,
                "error": str(e),
            }
        results.append(result)

    return results

# 1) Human agreement: Luiza vs Sara

In [30]:
def run_human_human_gamma():
    rows = []

    human_names = list(HUMAN_ANNOTATORS.keys())

    if len(human_names) != 2:
        print("This function assumes exactly two human annotators.")

    for task_type, labels in TASKS.items():
        for document_id in DOCUMENT_IDS:
            print(f"Human-human | {document_id} | {task_type}")

            all_annotations = []
            counts = {}

            for human_name in human_names:
                anns = load_human_annotations(document_id, human_name, labels)
                all_annotations.extend(anns)
                counts[f"{human_name}_annotations"] = len(anns)

            tests = gamma_suite(
                annotations=all_annotations,
                labels=labels,
                layer_name=task_type,
                expected_annotators=set(human_names),
            )

            for result in tests:
                result.update({
                    "agreement_type": "human-human",
                    "document": document_id,
                    "task_type": task_type,
                    "annotators": " vs ".join(human_names),
                    **counts,
                })
                rows.append(result)

    return pd.DataFrame(rows)


human_human_df = run_human_human_gamma()
human_human_df

Human-human | TED_001 | narration
Human-human | TED_002 | narration
Human-human | TED_004 | narration
Human-human | TED_005 | narration
Human-human | TED_006 | narration
Human-human | TED_007 | narration
Human-human | TED_008 | narration
Human-human | TED_010 | narration
Human-human | TED_011 | narration
Human-human | TED_012 | narration
Human-human | TED_015 | narration
Human-human | TED_016 | narration
Human-human | TED_017 | narration
Human-human | TED_018 | narration
Human-human | TED_019 | narration
Human-human | TED_020 | narration


,layer,gamma_type,alpha,beta,gamma,agreement_type,document,task_type,annotators,Luiza_annotations,Sara_annotations
0,narration,standard,1,1,0.072959,human-human,TED_001,narration,Luiza vs Sara,10,4
1,narration,soft,1,1,0.134532,human-human,TED_001,narration,Luiza vs Sara,10,4
2,narration,standard,1,2,0.160685,human-human,TED_001,narration,Luiza vs Sara,10,4
3,narration,soft,1,2,0.178139,human-human,TED_001,narration,Luiza vs Sara,10,4
4,narration,standard,1,1,0.242993,human-human,TED_002,narration,Luiza vs Sara,10,7
...,...,...,...,...,...,...,...,...,...,...,...
59,narration,soft,1,2,0.447827,human-human,TED_019,narration,Luiza vs Sara,18,10
60,narration,standard,1,1,-0.050609,human-human,TED_020,narration,Luiza vs Sara,23,5
61,narration,soft,1,1,-0.053333,human-human,TED_020,narration,Luiza vs Sara,23,5
62,narration,standard,1,2,-0.018882,human-human,TED_020,narration,Luiza vs Sara,23,5


# 2) Model-human agreement

Each model is compared separately to each human annotator.

In [34]:
def run_model_human_gamma():
    rows = []

    for task_type, labels in TASKS.items():
        for document_id in DOCUMENT_IDS:
            for model_label in MODELS.keys():
                print(f"Model-human | {document_id} | {task_type} | {model_label}")

                model_annotations = load_model_annotations(document_id, model_label, task_type, labels)

                for human_name in HUMAN_ANNOTATORS.keys():
                    human_annotations = load_human_annotations(document_id, human_name, labels)
                    all_annotations = human_annotations + model_annotations

                    expected_annotators = {human_name, model_label}

                    tests = gamma_suite(
                        annotations=all_annotations,
                        labels=labels,
                        layer_name=task_type,
                        expected_annotators=expected_annotators,
                    )

                    for result in tests:
                        result.update({
                            "agreement_type": "model-human",
                            "document": document_id,
                            "task_type": task_type,
                            "human_annotator": human_name,
                            "model": model_label,
                            "annotators": f"{model_label} vs {human_name}",
                            "human_annotations": len(human_annotations),
                            "model_annotations": len(model_annotations),
                        })
                        rows.append(result)

    return pd.DataFrame(rows)


model_human_df = run_model_human_gamma()
model_human_df

Model-human | TED_001 | narration | Mistral
Model-human | TED_001 | narration | Qwen
Model-human | TED_001 | narration | Gemma
Model-human | TED_002 | narration | Mistral
Model-human | TED_002 | narration | Qwen
Model-human | TED_002 | narration | Gemma
Model-human | TED_004 | narration | Mistral
Model-human | TED_004 | narration | Qwen
Model-human | TED_004 | narration | Gemma
Model-human | TED_005 | narration | Mistral
Model-human | TED_005 | narration | Qwen
Model-human | TED_005 | narration | Gemma
Model-human | TED_006 | narration | Mistral
Model-human | TED_006 | narration | Qwen
Model-human | TED_006 | narration | Gemma
Model-human | TED_007 | narration | Mistral
Model-human | TED_007 | narration | Qwen
Model-human | TED_007 | narration | Gemma
Model-human | TED_008 | narration | Mistral
Model-human | TED_008 | narration | Qwen
Model-human | TED_008 | narration | Gemma
Model-human | TED_010 | narration | Mistral
Model-human | TED_010 | narration | Qwen
Model-human | TED_010 | na

,layer,gamma_type,alpha,beta,gamma,agreement_type,document,task_type,human_annotator,model,annotators,human_annotations,model_annotations
0,narration,standard,1,1,0.390206,model-human,TED_001,narration,Luiza,Mistral,Mistral vs Luiza,10,5
1,narration,soft,1,1,0.496651,model-human,TED_001,narration,Luiza,Mistral,Mistral vs Luiza,10,5
2,narration,standard,1,2,0.478260,model-human,TED_001,narration,Luiza,Mistral,Mistral vs Luiza,10,5
3,narration,soft,1,2,0.566255,model-human,TED_001,narration,Luiza,Mistral,Mistral vs Luiza,10,5
4,narration,standard,1,1,0.141357,model-human,TED_001,narration,Sara,Mistral,Mistral vs Sara,4,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
379,narration,soft,1,2,0.333586,model-human,TED_020,narration,Luiza,Gemma,Gemma vs Luiza,23,10
380,narration,standard,1,1,0.053064,model-human,TED_020,narration,Sara,Gemma,Gemma vs Sara,5,10
381,narration,soft,1,1,0.051378,model-human,TED_020,narration,Sara,Gemma,Gemma vs Sara,5,10
382,narration,standard,1,2,0.072977,model-human,TED_020,narration,Sara,Gemma,Gemma vs Sara,5,10


# 3A) Model-model agreement: model pairs

This calculates:

- Mistral vs Qwen
- Mistral vs Gemma
- Qwen vs Gemma

In [35]:
def run_model_model_pair_gamma():
    rows = []

    for task_type, labels in TASKS.items():
        for document_id in DOCUMENT_IDS:
            for model_a, model_b in MODEL_PAIRS:
                print(f"Model-model pair | {document_id} | {task_type} | {model_a} vs {model_b}")

                annotations_a = load_model_annotations(document_id, model_a, task_type, labels)
                annotations_b = load_model_annotations(document_id, model_b, task_type, labels)

                all_annotations = annotations_a + annotations_b
                expected_annotators = {model_a, model_b}

                tests = gamma_suite(
                    annotations=all_annotations,
                    labels=labels,
                    layer_name=task_type,
                    expected_annotators=expected_annotators,
                )

                for result in tests:
                    result.update({
                        "agreement_type": "model-model-pair",
                        "document": document_id,
                        "task_type": task_type,
                        "model_a": model_a,
                        "model_b": model_b,
                        "annotators": f"{model_a} vs {model_b}",
                        "model_a_annotations": len(annotations_a),
                        "model_b_annotations": len(annotations_b),
                    })
                    rows.append(result)

    return pd.DataFrame(rows)


model_model_pair_df = run_model_model_pair_gamma()
model_model_pair_df

Model-model pair | TED_001 | narration | Mistral vs Qwen
Model-model pair | TED_001 | narration | Mistral vs Gemma
Model-model pair | TED_001 | narration | Qwen vs Gemma
Model-model pair | TED_002 | narration | Mistral vs Qwen
Model-model pair | TED_002 | narration | Mistral vs Gemma
Model-model pair | TED_002 | narration | Qwen vs Gemma
Model-model pair | TED_004 | narration | Mistral vs Qwen
Model-model pair | TED_004 | narration | Mistral vs Gemma
Model-model pair | TED_004 | narration | Qwen vs Gemma
Model-model pair | TED_005 | narration | Mistral vs Qwen
Model-model pair | TED_005 | narration | Mistral vs Gemma
Model-model pair | TED_005 | narration | Qwen vs Gemma
Model-model pair | TED_006 | narration | Mistral vs Qwen
Model-model pair | TED_006 | narration | Mistral vs Gemma
Model-model pair | TED_006 | narration | Qwen vs Gemma
Model-model pair | TED_007 | narration | Mistral vs Qwen
Model-model pair | TED_007 | narration | Mistral vs Gemma
Model-model pair | TED_007 | narrat

,layer,gamma_type,alpha,beta,gamma,agreement_type,document,task_type,model_a,model_b,annotators,model_a_annotations,model_b_annotations
0,narration,standard,1,1,0.640683,model-model-pair,TED_001,narration,Mistral,Qwen,Mistral vs Qwen,5,7
1,narration,soft,1,1,0.790064,model-model-pair,TED_001,narration,Mistral,Qwen,Mistral vs Qwen,5,7
2,narration,standard,1,2,0.737166,model-model-pair,TED_001,narration,Mistral,Qwen,Mistral vs Qwen,5,7
3,narration,soft,1,2,0.843672,model-model-pair,TED_001,narration,Mistral,Qwen,Mistral vs Qwen,5,7
4,narration,standard,1,1,0.514240,model-model-pair,TED_001,narration,Mistral,Gemma,Mistral vs Gemma,5,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,narration,soft,1,2,0.300180,model-model-pair,TED_020,narration,Mistral,Gemma,Mistral vs Gemma,5,10
188,narration,standard,1,1,0.264798,model-model-pair,TED_020,narration,Qwen,Gemma,Qwen vs Gemma,9,10
189,narration,soft,1,1,0.257720,model-model-pair,TED_020,narration,Qwen,Gemma,Qwen vs Gemma,9,10
190,narration,standard,1,2,0.294538,model-model-pair,TED_020,narration,Qwen,Gemma,Qwen vs Gemma,9,10


# 3B) Model-model agreement: all 3 models together

This calculates Gamma across Mistral, Qwen, and Gemma at the same time.

In [36]:
def run_model_model_all3_gamma():
    rows = []
    all_model_labels = list(MODELS.keys())

    for task_type, labels in TASKS.items():
        for document_id in DOCUMENT_IDS:
            print(f"Model-model all 3 | {document_id} | {task_type} | {', '.join(all_model_labels)}")

            all_annotations = []
            counts = {}

            for model_label in all_model_labels:
                anns = load_model_annotations(document_id, model_label, task_type, labels)
                all_annotations.extend(anns)
                counts[f"{model_label}_annotations"] = len(anns)

            tests = gamma_suite(
                annotations=all_annotations,
                labels=labels,
                layer_name=task_type,
                expected_annotators=set(all_model_labels),
            )

            for result in tests:
                result.update({
                    "agreement_type": "model-model-all3",
                    "document": document_id,
                    "task_type": task_type,
                    "annotators": " vs ".join(all_model_labels),
                    **counts,
                })
                rows.append(result)

    return pd.DataFrame(rows)


model_model_all3_df = run_model_model_all3_gamma()
model_model_all3_df

Model-model all 3 | TED_001 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_002 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_004 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_005 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_006 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_007 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_008 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_010 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_011 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_012 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_015 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_016 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_017 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_018 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_019 | narration | Mistral, Qwen, Gemma
Model-model all 3 | TED_020 | narration | Mistral, Qwen

,layer,gamma_type,alpha,beta,gamma,agreement_type,document,task_type,annotators,Mistral_annotations,Qwen_annotations,Gemma_annotations
0,narration,standard,1,1,0.687293,model-model-all3,TED_001,narration,Mistral vs Qwen vs Gemma,5,7,8
1,narration,soft,1,1,0.867150,model-model-all3,TED_001,narration,Mistral vs Qwen vs Gemma,5,7,8
2,narration,standard,1,2,0.760338,model-model-all3,TED_001,narration,Mistral vs Qwen vs Gemma,5,7,8
3,narration,soft,1,2,0.898431,model-model-all3,TED_001,narration,Mistral vs Qwen vs Gemma,5,7,8
4,narration,standard,1,1,0.359582,model-model-all3,TED_002,narration,Mistral vs Qwen vs Gemma,6,8,7
...,...,...,...,...,...,...,...,...,...,...,...,...
59,narration,soft,1,2,0.179142,model-model-all3,TED_019,narration,Mistral vs Qwen vs Gemma,27,7,28
60,narration,standard,1,1,0.316358,model-model-all3,TED_020,narration,Mistral vs Qwen vs Gemma,5,9,10
61,narration,soft,1,1,0.312684,model-model-all3,TED_020,narration,Mistral vs Qwen vs Gemma,5,9,10
62,narration,standard,1,2,0.336291,model-model-all3,TED_020,narration,Mistral vs Qwen vs Gemma,5,9,10


# Combine and save all results

In [37]:
all_gamma_results_df = pd.concat(
    [
        human_human_df,
        model_human_df,
        model_model_pair_df,
        model_model_all3_df,
    ],
    ignore_index=True,
    sort=False,
)

all_gamma_results_df

,layer,gamma_type,alpha,beta,gamma,agreement_type,document,task_type,annotators,Luiza_annotations,...,model,human_annotations,model_annotations,model_a,model_b,model_a_annotations,model_b_annotations,Mistral_annotations,Qwen_annotations,Gemma_annotations
0,narration,standard,1,1,0.072959,human-human,TED_001,narration,Luiza vs Sara,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,narration,soft,1,1,0.134532,human-human,TED_001,narration,Luiza vs Sara,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,narration,standard,1,2,0.160685,human-human,TED_001,narration,Luiza vs Sara,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,narration,soft,1,2,0.178139,human-human,TED_001,narration,Luiza vs Sara,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,narration,standard,1,1,0.242993,human-human,TED_002,narration,Luiza vs Sara,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699,narration,soft,1,2,0.179142,model-model-all3,TED_019,narration,Mistral vs Qwen vs Gemma,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.0,7.0,28.0
700,narration,standard,1,1,0.316358,model-model-all3,TED_020,narration,Mistral vs Qwen vs Gemma,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,9.0,10.0
701,narration,soft,1,1,0.312684,model-model-all3,TED_020,narration,Mistral vs Qwen vs Gemma,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,9.0,10.0
702,narration,standard,1,2,0.336291,model-model-all3,TED_020,narration,Mistral vs Qwen vs Gemma,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,9.0,10.0


## Save separate CSV files and one combined CSV file

In [38]:
human_human_path = OUTPUT_ROOT / "01_human_human_gamma_NARR.csv"
model_human_path = OUTPUT_ROOT / "02_model_human_gamma_NARR.csv"
model_model_pair_path = OUTPUT_ROOT / "03_model_model_pair_gamma_NARR.csv"
model_model_all3_path = OUTPUT_ROOT / "04_model_model_all3_gamma_NARR.csv"
all_results_path = OUTPUT_ROOT / "NARR_ALL_gamma_results.csv"

human_human_df.to_csv(human_human_path, index=False)
model_human_df.to_csv(model_human_path, index=False)
model_model_pair_df.to_csv(model_model_pair_path, index=False)
model_model_all3_df.to_csv(model_model_all3_path, index=False)
all_gamma_results_df.to_csv(all_results_path, index=False)

print("Saved files:")
print(human_human_path)
print(model_human_path)
print(model_model_pair_path)
print(model_model_all3_path)
print(all_results_path)

Saved files:
/content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/gamma_results/01_human_human_gamma_NARR.csv
/content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/gamma_results/02_model_human_gamma_NARR.csv
/content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/gamma_results/03_model_model_pair_gamma_NARR.csv
/content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/gamma_results/04_model_model_all3_gamma_NARR.csv
/content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/gamma_results/NARR_ALL_gamma_results.csv


## Optional summary table

This averages Gamma across documents for each agreement group.

In [39]:
summary_df = (
    all_gamma_results_df
    .groupby(["agreement_type", "task_type", "annotators", "gamma_type", "alpha", "beta"], dropna=False)["gamma"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values(["agreement_type", "task_type", "annotators", "gamma_type", "alpha", "beta"])
)

summary_path = OUTPUT_ROOT / "SUMMARY_average_gamma_by_agreement_type.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved summary to:", summary_path)
summary_df

Saved summary to: /content/drive/MyDrive/Zip files/ALL_GAMMA_RESULTS/NARR_Gamma/FINAL/gamma_results/SUMMARY_average_gamma_by_agreement_type.csv


,agreement_type,task_type,annotators,gamma_type,alpha,beta,mean,std,count
0,human-human,narration,Luiza vs Sara,soft,1,1,0.104744,0.482711,16
1,human-human,narration,Luiza vs Sara,soft,1,2,0.191810,0.336264,16
2,human-human,narration,Luiza vs Sara,standard,1,1,0.034175,0.436368,16
3,human-human,narration,Luiza vs Sara,standard,1,2,0.131986,0.285566,16
4,model-human,narration,Gemma vs Luiza,soft,1,1,0.334167,0.177960,16
5,model-human,narration,Gemma vs Luiza,soft,1,2,0.357356,0.200385,16
6,model-human,narration,Gemma vs Luiza,standard,1,1,0.277699,0.165491,16
7,model-human,narration,Gemma vs Luiza,standard,1,2,0.307387,0.182283,16
8,model-human,narration,Gemma vs Sara,soft,1,1,0.166238,0.237283,16
9,model-human,narration,Gemma vs Sara,soft,1,2,0.182471,0.266740,16


## Optional: inspect missing/failed calculations

If anything is missing, this cell shows rows where Gamma failed.

In [40]:
if "error" in all_gamma_results_df.columns:
    failed_df = all_gamma_results_df[all_gamma_results_df["error"].notna()]
    display(failed_df)
else:
    print("No error column found. That usually means all Gamma calculations succeeded.")

No error column found. That usually means all Gamma calculations succeeded.
